In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report

In [ ]:
# FIX: Use a relative path or update this to your actual dataset path
# Recommended dataset: https://www.kaggle.com/datasets/uciml/zoo-animal-classification
df = pd.read_csv("zoo.csv")  # <-- Update this path to your actual CSV file

# Preview the data
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# FIX: Replace 'class_type' with your actual target column name after previewing df.columns above
TARGET_COLUMN = 'class_type'  # <-- Update this to your actual target column name

# Drop non-feature columns (e.g. animal name column if present)
# FIX: Drop 'animal_name' only if it exists in the dataset
drop_cols = [TARGET_COLUMN]
if 'animal_name' in df.columns:
    drop_cols.append('animal_name')

X = df.drop(columns=drop_cols)
y = df[TARGET_COLUMN]

# FIX: Encode target if it's categorical (string labels)
if y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)
    print("Classes:", le.classes_)

print("Features shape:", X.shape)
print("Target distribution:\n", pd.Series(y).value_counts())

In [ ]:
# FIX: Added train/test split BEFORE scaling (was missing entirely)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # FIX: use transform (not fit_transform) on test set

# FIX: Use RandomForestClassifier instead of RandomForestRegressor (species = categorical)
model = RandomForestClassifier(random_state=42)

# Train the model
model.fit(X_train_scaled, y_train)

print("Model training completed.")

In [ ]:
# FIX: Removed 'auto' from max_features — not valid in scikit-learn >= 1.1
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [ ]:
# FIX: scoring changed to 'accuracy' to match classifier; model now defined above
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',  # FIX: was 'r2' which is for regression
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

best_params = grid_search.best_params_
print('Best Parameters:', best_params)

# Train the best model
best_model = grid_search.best_estimator_

# Predict on test set
y_pred = best_model.predict(X_test_scaled)

In [ ]:
# FIX: Removed MSE/R2 (regression metrics) — replaced with classification metrics
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Feature importance plot
importances = best_model.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.title('Feature Importances — Animal Species Identifier')
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.tight_layout()
plt.show()